# Iris Flower Classification

This notebook demonstrates a complete machine learning workflow for classifying Iris flowers into three species (setosa, versicolor, virginica) based on their physical measurements.

## Dataset Features:
- **SepalLengthCm**: Sepal length in centimeters
- **SepalWidthCm**: Sepal width in centimeters
- **PetalLengthCm**: Petal length in centimeters
- **PetalWidthCm**: Petal width in centimeters
- **Species**: Target variable (Iris-setosa, Iris-versicolor, Iris-virginica)

## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 2: Load and Explore the Data

In [ ]:
# Load the Iris dataset
df = pd.read_csv('Iris.csv')

# Display basic information
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nBasic statistics:")
print(df.describe())

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print("\nSpecies distribution:")
print(df['Species'].value_counts())

## Step 3: Data Visualization

In [ ]:
# Create a figure with multiple subplots for exploration
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot distributions of features by species
for idx, col in enumerate(['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']):
    ax = axes[idx // 2, idx % 2]
    for species in df['Species'].unique():
        data = df[df['Species'] == species][col]
        ax.hist(data, alpha=0.6, label=species)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {col}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Create a correlation heatmap
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for each feature by species
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, col in enumerate(['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']):
    ax = axes[idx // 2, idx % 2]
    sns.boxplot(data=df, x='Species', y=col, ax=ax, palette='Set2')
    ax.set_title(f'Box Plot of {col}')

plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# Separate features and target
X = df[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
y = df['Species']

# Encode the target variable (convert species names to numbers)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y_encoded.shape)
print("\nClass mapping:")
for i, species in enumerate(label_encoder.classes_):
    print(f"{i}: {species}")

## Step 5: Train-Test Split

In [ ]:
# Split the data into training and testing sets
# 80% training, 20% testing with random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u} ({label_encoder.classes_[u]}): {c}")

## Step 6: Model Training

In [ ]:
# Train a Random Forest Classifier
# Random Forest is an ensemble method that combines multiple decision trees
model = RandomForestClassifier(
    n_estimators=100,  # Number of trees in the forest
    random_state=42,
    n_jobs=-1  # Use all available processors
)

# Fit the model to the training data
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Number of trees: {model.n_estimators}")

## Step 7: Model Predictions

In [ ]:
# Make predictions on training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Get prediction probabilities
y_test_pred_proba = model.predict_proba(X_test)

print("Predictions generated successfully!")
print(f"\nFirst 10 predictions (test set):")
for i in range(10):
    predicted_class = label_encoder.classes_[y_test_pred[i]]
    actual_class = label_encoder.classes_[y_test[i]]
    confidence = y_test_pred_proba[i].max() * 100
    match = '✓' if y_test_pred[i] == y_test[i] else '✗'
    print(f"{match} Predicted: {predicted_class}, Actual: {actual_class}, Confidence: {confidence:.2f}%")

## Step 8: Model Evaluation

In [ ]:
# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"\nTraining Accuracy: {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy: {test_accuracy * 100:.2f}%")

# Calculate additional metrics
precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')
f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nWeighted Precision: {precision:.4f}")
print(f"Weighted Recall: {recall:.4f}")
print(f"Weighted F1-Score: {f1:.4f}")

In [ ]:
# Detailed classification report
print("\n" + "=" * 50)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print(cm)

## Step 9: Feature Importance Analysis

In [ ]:
# Get feature importances from the Random Forest model
feature_names = X.columns
feature_importance = model.feature_importances_

# Create a sorted dataframe
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(importance_df.to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(10, 6))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Feature Importance in Random Forest Model')
plt.gca().invert_yaxis()

# Add value labels on bars
for bar in bars:
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
             ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Step 10: Model Summary and Insights

In [ ]:
print("=" * 60)
print("IRIS FLOWER CLASSIFICATION - MODEL SUMMARY")
print("=" * 60)

print("\n1. DATASET INFORMATION:")
print(f"   - Total samples: {len(df)}")
print(f"   - Features: {len(X.columns)}")
print(f"   - Classes: {len(label_encoder.classes_)}")
print(f"   - Training samples: {len(X_train)}")
print(f"   - Testing samples: {len(X_test)}")

print("\n2. MODEL SELECTION:")
print(f"   - Algorithm: Random Forest Classifier")
print(f"   - Number of trees: {model.n_estimators}")
print(f"   - Why Random Forest?")
print(f"     • Handles both linear and non-linear relationships")
print(f"     • Robust to outliers")
print(f"     • Provides feature importance rankings")
print(f"     • Excellent for multi-class classification")

print("\n3. PERFORMANCE METRICS:")
print(f"   - Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall: {recall:.4f}")
print(f"   - F1-Score: {f1:.4f}")

print("\n4. KEY FINDINGS:")
most_important_feature = importance_df.iloc[0]
least_important_feature = importance_df.iloc[-1]
print(f"   - Most important feature: {most_important_feature['Feature']} ({most_important_feature['Importance']:.4f})")
print(f"   - Least important feature: {least_important_feature['Feature']} ({least_important_feature['Importance']:.4f})")

errors = (y_test != y_test_pred).sum()
print(f"   - Misclassifications on test set: {errors}/{len(y_test)}")

print("\n5. CLASSIFICATION CONCEPTS LEARNED:")
print("   - Supervised Learning: Using labeled data to train the model")
print("   - Multi-class Classification: Predicting one of 3+ classes")
print("   - Train-Test Split: Evaluating on unseen data to detect overfitting")
print("   - Feature Engineering: Using raw measurements as features")
print("   - Model Evaluation: Using metrics like accuracy, precision, recall")
print("   - Confusion Matrix: Understanding types of errors")
print("   - Feature Importance: Identifying which features drive predictions")

print("\n" + "=" * 60)

## Step 11: Making Predictions on New Data

In [ ]:
# Example: Predict species for a new flower sample
new_sample = np.array([[5.5, 3.5, 1.5, 0.5]])  # [SepalLen, SepalWidth, PetalLen, PetalWidth]

prediction = model.predict(new_sample)[0]
prediction_proba = model.predict_proba(new_sample)[0]
predicted_species = label_encoder.classes_[prediction]

print("Prediction for new sample:")
print(f"Measurements: SepalLength=5.5cm, SepalWidth=3.5cm, PetalLength=1.5cm, PetalWidth=0.5cm")
print(f"\nPredicted Species: {predicted_species}")
print(f"\nProbabilities for each class:")
for i, species in enumerate(label_encoder.classes_):
    print(f"  {species}: {prediction_proba[i] * 100:.2f}%")